In [6]:
# Import required packages 
from matplotlib import pyplot as plt 
from src.show_img import show_images
import cv2 as cv 
import numpy as np 
import pandas as pd 
import os 
import datetime 

In [7]:
# Window config 
window_name = "Attendance system" 
cv.namedWindow(window_name, cv.WINDOW_NORMAL) 

In [8]:
# Models config     
face_detector = cv.FaceDetectorYN.create( # face detector model 
    "assets/model/face_detection_yunet_2022mar.onnx", 
    "", 
    (320, 320), 
    0.9, 
    0.3, 
    5000
)

face_recognizer = cv.FaceRecognizerSF.create( # face recognition model 
    "assets/model/face_recognition_sface_2021dec.onnx", 
    ""
)

In [9]:
# ---------------
# Capture config 
# ---------------

# Create a video capture 
capture = cv.VideoCapture(0) 

# Shape of capture 
capture_width = int(capture.get(cv.CAP_PROP_FRAME_WIDTH))
capture_height = int(capture.get(cv.CAP_PROP_FRAME_HEIGHT))

In [10]:
# Frame configuration 

l2_threshold = 1.128

student_features = {} 
db_path = "assets/DB" 
for filename in os.listdir(db_path):
    if filename.lower().endswith(".jpg"): 
        image_path = os.path.join(db_path, filename)
        student_name = os.path.splitext(filename)[0]

        image = cv.imread(image_path) 
        if image is None:
            print(f"Could not read image {image_path}") 
            continue

        face_detector.setInputSize((image.shape[1], image.shape[0]))
        faces = face_detector.detect(image)

        if faces[1] is not None and len(faces[1]) > 0:
            face = faces[1][0]
            frame_align_crop = face_recognizer.alignCrop(image, face)
            feature = face_recognizer.feature(frame_align_crop)
            student_features[student_name] = feature
            print(f"Loaded: {student_name}")
        else:
            print(f"Warning: No face detected in {filename}. Skipping.")
print(f"Database loaded with {len(student_features)} students.")



attendance_log = []
logged_today = {} 

while True: 
    ret, frame = capture.read() 
    frame = cv.flip(frame, 1) 

    # Set input size for capture 
    face_detector.setInputSize((capture_width, capture_height)) 

    # Detect 
    frame_faces = face_detector.detect(frame) 

    current_date_str = datetime.datetime.now().strftime("%Y-%m-%d")

    if frame_faces[1] is not None: 
        for face in frame_faces[1]: 
            coordinates = face[:-1].astype(np.int32) 
            x, y, w, h = coordinates[0], coordinates[1], coordinates[2], coordinates[3] 

            frame_align_crop = face_recognizer.alignCrop(frame, face)

            try: 
                frame_feature = face_recognizer.feature(frame_align_crop)
            except Exception as e: 
                print(f"Error extracting with {e}") 
                continue

            best_match_name = "Unknown"
            min_l2_score = float('inf') 

            for student_name, student_feature in student_features.items(): 
                l2_score = face_recognizer.match(frame_feature, student_feature, cv.FACE_RECOGNIZER_SF_FR_NORM_L2) 

                if l2_score < min_l2_score: 
                    min_l2_score = l2_score 
                    if l2_score <= l2_threshold: 
                        best_match_name = student_name 
                    else: 
                        best_match_name = "Unknown"

            current_time = datetime.datetime.now() 
            log_time_obj = current_time.date()
            timestamp_str = current_time.strftime("%Y-%m-%d %H:%M:%S")
            if best_match_name != "Unknown": 
                if best_match_name in logged_today and logged_today[best_match_name] == current_date_str: 


                    already_logged = False
                    for entry in attendance_log:
                        log_time = datetime.datetime.strptime(entry["Timestamp"], "%Y-%m-%d %H:%M:%S")
                        if entry["Name"] == best_match_name and (current_time - log_time).total_seconds() < 10:
                            already_logged = True
                            break

                else:
                    attendance_log.append({
                        "Name": best_match_name,
                        "Status": "Present", 
                        "Date": log_time_obj,
                        "Timestamp": timestamp_str,
                    })
                    logged_today[best_match_name] = current_date_str
                    print(f"Logged: {best_match_name} at {timestamp_str}")


            color = None 
            if best_match_name != "Unknown": 
                color = (0, 255, 0) 
            else: 
                color = (0, 0, 255)

            cv.rectangle(frame, (x, y), (x + w, y + h), color, 3) 
            cv.putText(frame, best_match_name, (x, y - 10), cv.FONT_ITALIC, 0.8, color, 2, cv.LINE_AA)

    cv.imshow(window_name, frame) 
    key = cv.waitKey(1) 
    if key == ord("q"): 
        break 

cv.destroyAllWindows() 
capture.release() 

if attendance_log:
    print("Saving attendance log to attendance.xlsx...")
    df = pd.DataFrame(attendance_log)
    
    # Ensure Date column exists and is correct type
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    
    # Reorder columns for clarity
    df = df[["Date", "Name", "Status", "Timestamp"]]
    
    try:
        # Try to load existing data and append, keeping only the latest entry for each student per day
        try:
            existing_df = pd.read_excel("assets/output/attendance.xlsx")
            existing_df['Date'] = pd.to_datetime(existing_df['Date']).dt.date # Ensure date format consistency
            
            # Combine new and existing logs
            combined_df = pd.concat([existing_df, df])
            
            # Remove duplicates, keeping the earliest entry for each student on each day
            # This assumes the 'Timestamp' in attendance_log is the earliest for that day's first detection
            # If you want the *latest* detection of the day, logic needs slight adjustment
            combined_df = combined_df.sort_values(by="Timestamp") # Sort by timestamp to ensure earliest is kept
            combined_df = combined_df.drop_duplicates(subset=["Name", "Date"], keep="first") 
            
            df_to_save = combined_df
            print("Appended new entries and handled duplicates.")

        except FileNotFoundError:
            df_to_save = df # If file doesn't exist, just use the current log
            print("attendance.xlsx not found, creating a new file.")
        
        df_to_save.to_excel("assets/output/attendance.xlsx", index=False)
        print("Attendance log saved successfully.")
    except Exception as e:
        print(f"Error saving attendance log to Excel: {e}")
else:
    print("No attendance data to save.")

print("Process finished.")

Loaded: elon
Loaded: amirmohammad
Loaded: ehsan
Loaded: omid
Loaded: parisa
Loaded: hamid
Loaded: hassan
Loaded: abbas
Loaded: saeed
Loaded: javad
Loaded: hossein
Loaded: parham
Loaded: jafar
Loaded: ali
Database loaded with 14 students.
Logged: amirmohammad at 2026-04-11 21:10:05
Saving attendance log to attendance.xlsx...
Appended new entries and handled duplicates.
Attendance log saved successfully.
Process finished.
